In [ ]:
# 0,1: 00 for psi_1, 01 for psi_2 ...
# 2,3,4: For states
# 5,6: Ancillary qubits

In [ ]:
from qiskit import QuantumCircuit
from numpy import sqrt

def circuit_init():
    qc = QuantumCircuit(7)
    return qc

def PREP(qc):

    desired_vector = [
        1 / sqrt(3), 0, 0, 0, 0, 0, 0, 0,
        -1 / (8 * sqrt(3)), -1 / 8, -1 / 8, -sqrt(3) / 8,
        -1 / 8, -sqrt(3) / 8, -sqrt(3) / 8, -3 / 8,
        -1 / (8 * sqrt(3)), 1 / 8, 1 / 8, -sqrt(3) / 8,
        1 / 8, -sqrt(3) / 8, -sqrt(3) / 8, 3 / 8,
        0, 0, 0, 0, 0, 0, 0, 0
    ]
    
    qc.initialize(desired_vector, [4, 3, 2, 1, 0])
    return qc

In [ ]:
from pennylane.templates.state_preparations.mottonen import compute_theta, gray_code
from qiskit.circuit.library.standard_gates import RXGate,RYGate, RZGate
from qiskit.circuit.library import MCXGate
from numpy import array, log2

def RRR_X(qc, wires, params):
    qc.append(RXGate(params[0]),[wires[0]])
    qc.append(RXGate(params[1]),[wires[1]])
    qc.append(RXGate(params[2]),[wires[2]])
    
def RRR_Z(qc, wires, params):
    qc.append(RZGate(params[0]),[wires[0]])
    qc.append(RZGate(params[1]),[wires[1]])
    qc.append(RZGate(params[2]),[wires[2]])


def U_CCCR(qc,wires, params):
    qc.append(RYGate(params[0]).control(3, ctrl_state = '000'[::-1]), [wires[0], wires[1],wires[2],wires[3]])
    qc.append(RYGate(params[1]).control(3, ctrl_state = '001'[::-1]), [wires[0], wires[1],wires[2],wires[3]])
    qc.append(RYGate(params[2]).control(3, ctrl_state = '010'[::-1]), [wires[0], wires[1],wires[2],wires[3]])
    qc.append(RYGate(params[3]).control(3, ctrl_state = '011'[::-1]), [wires[0], wires[1],wires[2],wires[3]])
    qc.append(RYGate(params[4]).control(3, ctrl_state = '100'[::-1]), [wires[0], wires[1],wires[2],wires[3]])
    qc.append(RYGate(params[5]).control(3, ctrl_state = '101'[::-1]), [wires[0], wires[1],wires[2],wires[3]])
    qc.append(RYGate(params[6]).control(3, ctrl_state = '110'[::-1]), [wires[0], wires[1],wires[2],wires[3]])
    qc.append(RYGate(params[7]).control(3, ctrl_state = '111'[::-1]), [wires[0], wires[1],wires[2],wires[3]])

def U_CCCR_decom(qc,wires,params):
    # Make sure that control qubits should be increasing order
    params = array(params)
    params = compute_theta(params) #can be on/off

    num_Ucontrols = len(wires)-1
    code = gray_code(3)
    n_selections = len(code)
    control_order = [int(log2(int(code[i], 2) ^ int(code[(i + 1) % n_selections], 2))) for i in range(n_selections)]

    for i in range(n_selections):
        qc.ry(params[i],wires[3])
        qc.cx(wires[num_Ucontrols-1 - control_order[i]],wires[3])   
    return qc


def CRRR_X(qc, wires, state, params):
    qc.append(RXGate(params[0]).control(1,ctrl_state = state[::-1]), [wires[0], wires[1]])  # Control: wires[0] Target: wires[1]
    qc.append(RXGate(params[1]).control(1,ctrl_state = state[::-1]), [wires[0], wires[2]])  # Control: wires[0] Target: wires[2]
    qc.append(RXGate(params[2]).control(1,ctrl_state = state[::-1]), [wires[0], wires[3]])  # Control: wires[0] Target: wires[3]
    return qc

def CRRR_Z(qc, wires, state, params):
    qc.append(RZGate(params[0]).control(1,ctrl_state = state[::-1]), [wires[0], wires[1]])  # Control: wires[0] Target: wires[1]
    qc.append(RZGate(params[1]).control(1,ctrl_state = state[::-1]), [wires[0], wires[2]])  # Control: wires[0] Target: wires[2]
    qc.append(RZGate(params[2]).control(1,ctrl_state = state[::-1]), [wires[0], wires[3]])  # Control: wires[0] Target: wires[2]
    return qc

def U_CCCcR(qc,wires, params):
    qc.append(RYGate(params[0]).control(4, ctrl_state = '0001'[::-1]), [wires[0], wires[1],wires[2],wires[3], wires[4]])
    qc.append(RYGate(params[1]).control(4, ctrl_state = '0011'[::-1]), [wires[0], wires[1],wires[2],wires[3], wires[4]])
    qc.append(RYGate(params[2]).control(4, ctrl_state = '0101'[::-1]), [wires[0], wires[1],wires[2],wires[3], wires[4]])
    qc.append(RYGate(params[3]).control(4, ctrl_state = '0111'[::-1]), [wires[0], wires[1],wires[2],wires[3], wires[4]])
    qc.append(RYGate(params[4]).control(4, ctrl_state = '1001'[::-1]), [wires[0], wires[1],wires[2],wires[3], wires[4]])
    qc.append(RYGate(params[5]).control(4, ctrl_state = '1011'[::-1]), [wires[0], wires[1],wires[2],wires[3], wires[4]])
    qc.append(RYGate(params[6]).control(4, ctrl_state = '1101'[::-1]), [wires[0], wires[1],wires[2],wires[3], wires[4]])
    qc.append(RYGate(params[6]).control(4, ctrl_state = '1111'[::-1]), [wires[0], wires[1],wires[2],wires[3], wires[4]])
    #ctrl_state = '1111'[::-1] make divide 0 problem. 

def U_CCCcR_decom(qc,wires,params):
    # Make sure that control qubits should be increasing order
    params = array(params)
    params = compute_theta(params) #can be on/off

    num_Ucontrols = len(wires)-2
    code = gray_code(3)
    n_selections = len(code)
    control_order = [int(log2(int(code[i], 2) ^ int(code[(i + 1) % n_selections], 2))) for i in range(n_selections)]

    for i in range(n_selections):
        qc.cry(params[i],wires[3],wires[4])
        CCX(qc,[wires[num_Ucontrols-1 - control_order[i]],wires[3],wires[4]],'11')
    
    return qc

def CCRRR_X(qc, wires, state, params):
    qc.append(RXGate(params[0]).control(2, ctrl_state = state[::-1]), [wires[0], wires[1],wires[2]])  # Control: wires[0] wires[1]
    qc.append(RXGate(params[1]).control(2, ctrl_state = state[::-1]), [wires[0], wires[1],wires[3]])  # Control: wires[0] wires[1]
    qc.append(RXGate(params[2]).control(2, ctrl_state = state[::-1]), [wires[0], wires[1],wires[4]])  # Control: wires[0] wires[1]
    return qc

def CCRRR_Y(qc, wires, state, params):
    qc.append(RYGate(params[0]).control(2, ctrl_state = state[::-1]), [wires[0], wires[1],wires[2]])  # Control: wires[0] wires[1]
    qc.append(RYGate(params[1]).control(2, ctrl_state = state[::-1]), [wires[0], wires[1],wires[3]])  # Control: wires[0] wires[1]
    qc.append(RYGate(params[2]).control(2, ctrl_state = state[::-1]), [wires[0], wires[1],wires[4]])  # Control: wires[0] wires[1]
    return qc

def CCX(qc, wires, state):
    mcx_gate = MCXGate(num_ctrl_qubits=2,ctrl_state=state[::-1])
    qc.append(mcx_gate, [wires[0], wires[1],wires[2]])
    return qc

def CCCX(qc, wires, state):
    mcx_gate = MCXGate(num_ctrl_qubits=3,ctrl_state=state[::-1])
    qc.append(mcx_gate, [wires[0], wires[1],wires[2], wires[3]])
    return qc

In [ ]:
from qiskit.circuit import ParameterVector

def Ansatz(qc, wires):

    params = ParameterVector('x', 46)

    #Big made by HEA 
    RRR_X(qc,wires[0:3],params[0:3])
    RRR_Z(qc,wires[0:3],params[3:6])
    qc.cx(wires[0],wires[1])
    qc.cx(wires[1],wires[2])

    #Module 1
    U_CCCR_decom(qc,wires[0:4],params[6:14])
    CRRR_X(qc,[wires[3],wires[0],wires[1],wires[2]],'0',params[14:17])
    CRRR_Z(qc,[wires[3],wires[0],wires[1],wires[2]],'0',params[17:20])
    CCX(qc,[wires[0],wires[3], wires[1]],'10')
    CCX(qc,[wires[1],wires[3], wires[2]],'10')

    CRRR_X(qc,[wires[3],wires[0],wires[1],wires[2]],'1',params[20:23])
    CRRR_Z(qc,[wires[3],wires[0],wires[1],wires[2]],'1',params[23:26])
    CCX(qc,[wires[0],wires[3], wires[1]],'11')
    CCX(qc,[wires[1],wires[3], wires[2]],'11')

    #Module 2
    U_CCCcR_decom(qc, wires[0:5],params[26:34])
    CCRRR_X(qc,[wires[3],wires[4],wires[0],wires[1],wires[2]],'10',params[34:37])
    CCRRR_Y(qc,[wires[3],wires[4],wires[0],wires[1],wires[2]],'10',params[37:40])
    CCCX(qc,[wires[0],wires[3],wires[4],wires[1]],'110')
    CCCX(qc,[wires[1],wires[3],wires[4],wires[2]],'110')
    CCRRR_X(qc,[wires[3],wires[4],wires[0],wires[1],wires[2]],'11',params[40:43])
    CCRRR_Y(qc,[wires[3],wires[4],wires[0],wires[1],wires[2]],'11',params[43:46])
    CCCX(qc,[wires[0],wires[3],wires[4],wires[1]],'111')
    CCCX(qc,[wires[1],wires[3],wires[4],wires[2]],'111')    

    qc.measure_all()
    return qc

In [ ]:
from numpy.random import rand
num_params=46
qc = circuit_init()
PREP(qc)
Ansatz(qc, [2,3,4,5,6])
qc_reversed=qc.reverse_bits()
qc.draw(output='mpl', fold=False, style = 'clifford') 

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

service = QiskitRuntimeService(channel='ibm_quantum')
#backend = service.least_busy(min_num_qubits=127)
backend = service.backend("ibm_strasbourg")
print(backend)

pm = generate_preset_pass_manager(optimization_level=3,backend=backend)


candidate_circuit = pm.run(qc_reversed)
candidate_circuit.draw('mpl', fold=False, scale=0.1, idle_wires=False)

In [ ]:
from qiskit.quantum_info import SparsePauliOp

# Define the factor (I + Z)/2 = |0><0|
o_term = SparsePauliOp.from_list([("I", 0.5), ("Z", 0.5)])

# Define the factor (I - Z)/2 = |1><1|
l_term = SparsePauliOp.from_list([("I", 0.5), ("Z", -0.5)])

I = SparsePauliOp.from_list([("I", 1)])

# Use tensor products to create (I + Z)/2 ⊗ (I + Z)/2 ⊗ (I + Z)/2 ⊗ (I + Z)/2
Ob1= o_term.tensor(o_term).tensor(I).tensor(I).tensor(I).tensor(o_term).tensor(o_term)
# Check if b0b1 = 01, Check if b3b4 = 10
Ob2= o_term.tensor(l_term).tensor(I).tensor(I).tensor(I).tensor(l_term).tensor(o_term)
# Check if b0b1 = 10, Check if b3b4 = 11
Ob3= l_term.tensor(o_term).tensor(I).tensor(I).tensor(I).tensor(l_term).tensor(l_term)

cost_hamiltonian = Ob1 + Ob2 + Ob3
cost_hamiltonian=cost_hamiltonian.apply_layout(candidate_circuit.layout)

In [ ]:
def cost_func_estimator(params, ansatz, hamiltonian, estimator):
    # Prepare the job input with the ansatz, Hamiltonian, and parameters
    pub = (ansatz, hamiltonian, params)
    job = estimator.run([pub])

    # Retrieve the result
    results = job.result()[0]

    # Extract the cost (expectation value)
    cost = results.data.evs
    print('cost is', cost)

    # Append cost to the global success probability list and return it
    success_probability.append(cost)
    return 1/cost

In [ ]:
from qiskit_ibm_runtime import Session, EstimatorV2 as Estimator
from scipy.optimize import minimize

init_params = rand(num_params)
success_probability = [] # Global variable

with Session(backend=backend) as session:

    estimator = Estimator(mode=session)
    estimator.options.default_shots = 1000

    result = minimize(
        cost_func_estimator,
        init_params,
        args=(candidate_circuit, cost_hamiltonian, estimator),
        method="COBYLA",
        tol=1e-1,
    )
    print(result)
    print(result.x)

In [ ]:
import matplotlib.pyplot as plt
plt.plot(success_probability, label="Sucess Probability")
plt.xlabel('Iterations')
plt.ylabel('Sucess Probability')
plt.legend()
plt.show()

In [ ]:
import openpyxl

# create a new workbook
workbook = openpyxl.Workbook()

# select the active worksheet
worksheet = workbook.active

# loop through the confidences array and write values to the worksheet
for i in range(len(success_probability)):
    worksheet.cell(row=i+1, column=1, value=float(success_probability[i]))

# save the workbook to a file
workbook.save('ME Yordan+HEA TT states tol01 (ibm_strasbourg).xlsx')

In [ ]:
from numpy import max
max(success_probability)